<div style='background: #0f1f3d; padding: 40px 48px 36px; border-radius: 6px;'>
  <p style='color: #5a9bd5; margin: 0 0 16px 0; font-size: 0.72em; letter-spacing: 3px; text-transform: uppercase; font-weight: 500;'>FARMSA Research Group &mdash; Module 6</p>
  <h1 style='color: #ffffff; font-size: 2em; font-weight: 600; margin: 0 0 6px 0; letter-spacing: -0.5px;'>Portfolio Optimizer Tool</h1>
  <p style='color: #8fadc8; font-size: 0.95em; font-weight: 400; margin: 12px 0 0 0;'>Interactive Mean-Variance Optimization with Multiple Covariance Estimators</p>
</div>

---
## How to Use This Tool

1. **Run all cells** in order (Cell → Run All).
2. **Select your preferences** in the control panel below: pick an estimator, adjust risk tolerance, choose stocks, set constraints.
3. **Click "Optimize"** to compute the optimal portfolio.
4. **Read the output**: weights table, allocation chart, and efficient frontier.

No financial background required — hover over any (?) label for a plain-English explanation.

In [1]:
# ── §1  Setup & Data ────────────────────────────────────────
%pip install -q scipy yfinance scikit-learn ipywidgets
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings, sys, os
from scipy.optimize import minimize
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
warnings.filterwarnings('ignore')

# ── Load the shared 50-stock universe ──
prices  = pd.read_csv('data/prices.csv',  index_col=0, parse_dates=True)
returns = pd.read_csv('data/returns.csv', index_col=0, parse_dates=True)
tickers = list(returns.columns)
N = len(tickers)

# ── Sector mapping ──
# Generated from yfinance via scripts/build_metadata.py.
# To refresh: run  python scripts/build_metadata.py
_meta = pd.read_csv('data/metadata.csv', index_col='ticker')
SECTOR_MAP = _meta['sector'].to_dict()
SECTORS = sorted(_meta['sector'].unique())

# ── Build sector → tickers lookup ──
sector_tickers = {}
for t in tickers:
    sec = SECTOR_MAP.get(t, 'Other')
    sector_tickers.setdefault(sec, []).append(t)

print(f"✓ Loaded {N} assets × {returns.shape[0]} days")
print(f"  Sectors: {', '.join(SECTORS)}")


[notice] A new release of pip is available: 23.3.1 -> 26.0.1
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✓ Loaded 50 assets × 1759 days
  Sectors: Communication, Cons. Discr., Cons. Staples, Energy, Financials, Healthcare, Industrials, Materials, Real Estate, Technology, Utilities


In [2]:
# ── §2  Estimator Registry ──────────────────────────────────
# Import from estimators.py (lives alongside this notebook).
# Each estimator: returns_df (T×N DataFrame) → N×N numpy array.
#
# To update an estimator, edit estimators.py — no changes needed here.

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
from estimators import ESTIMATORS

estimator_names = list(ESTIMATORS.keys())
print(f"✓ {len(ESTIMATORS)} estimators loaded: {', '.join(estimator_names)}")

✓ 5 estimators loaded: Sample Covariance, Ledoit-Wolf Shrinkage, RMT Eigenvalue Cleaning, Fama-French 3-Factor, DCC-GARCH


In [3]:
# ── §3  Optimization Engine ─────────────────────────────────

def estimate_mu(returns_df):
    """Annualized expected returns (historical mean)."""
    return returns_df.mean().values * 252

def optimize_portfolio(mu, cov, risk_tolerance, max_weight=1.0,
                       sector_limit=1.0, selected_tickers=None,
                       all_tickers=None):
    """
    Mean-variance optimization with constraints.
    
    risk_tolerance: 0 = min variance, 1 = max Sharpe, intermediate = blend.
    max_weight:     upper bound per position (e.g., 0.10 = 10%).
    sector_limit:   max total weight per sector (e.g., 0.30 = 30%).
    selected_tickers: subset of tickers to include.
    all_tickers:    full ticker list (for indexing).
    
    Returns dict with weights, expected return, expected vol, Sharpe.
    """
    if all_tickers is None:
        all_tickers = tickers
    
    # Map selected tickers to indices
    if selected_tickers is not None:
        idx_map = [all_tickers.index(t) for t in selected_tickers if t in all_tickers]
    else:
        idx_map = list(range(len(all_tickers)))
    
    n = len(idx_map)
    if n == 0:
        return None
    
    mu_sub  = mu[idx_map]
    cov_sub = cov[np.ix_(idx_map, idx_map)]
    
    # Regularize if needed
    min_eig = np.min(np.linalg.eigvalsh(cov_sub))
    if min_eig < 1e-10:
        cov_sub += np.eye(n) * (1e-10 - min_eig)
    
    w0 = np.ones(n) / n
    bounds = [(0, min(max_weight, 1.0))] * n
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
    
    # Sector constraints
    if sector_limit < 1.0:
        sub_tickers = [all_tickers[i] for i in idx_map]
        for sec in SECTORS:
            sec_idx = [j for j, t in enumerate(sub_tickers) if SECTOR_MAP.get(t) == sec]
            if len(sec_idx) > 0:
                constraints.append({
                    'type': 'ineq',
                    'fun': lambda w, si=sec_idx, sl=sector_limit: sl - np.sum(w[si])
                })
    
    # Objective: blend between min-variance and max-Sharpe
    # λ=0 → min variance, λ=1 → max Sharpe (via max utility w'μ - 0.5·δ·w'Σw)
    rf_annual = 0.04  # approximate risk-free rate
    
    if risk_tolerance < 0.01:
        # Pure min variance
        def objective(w):
            return w @ cov_sub @ w
    else:
        # Mean-variance utility: maximize w'μ - (δ/2) w'Σw
        # Higher risk_tolerance → lower δ → more aggressive
        delta = 10.0 * (1.0 - risk_tolerance) + 0.5 * risk_tolerance
        def objective(w):
            return -(w @ mu_sub - 0.5 * delta * w @ cov_sub @ w)
    
    res = minimize(objective, w0, method='SLSQP', bounds=bounds,
                   constraints=constraints, options={'ftol': 1e-12, 'maxiter': 2000})
    
    w_opt = res.x if res.success else w0
    w_opt = np.maximum(w_opt, 0)
    w_opt /= w_opt.sum()
    
    # Portfolio stats (annualized)
    port_ret = w_opt @ mu_sub
    port_vol = np.sqrt(w_opt @ cov_sub @ w_opt * 252)
    sharpe   = (port_ret - rf_annual) / port_vol if port_vol > 0 else 0
    
    # Map back to full ticker list
    full_weights = np.zeros(len(all_tickers))
    for j, i in enumerate(idx_map):
        full_weights[i] = w_opt[j]
    
    return {
        'weights': full_weights,
        'tickers': all_tickers,
        'expected_return': port_ret,
        'expected_vol': port_vol,
        'sharpe': sharpe,
        'success': res.success,
    }


def compute_efficient_frontier(mu, cov, selected_idx, max_weight=1.0,
                               sector_limit=1.0, n_points=30):
    """Compute efficient frontier for the selected assets."""
    n = len(selected_idx)
    mu_sub  = mu[selected_idx]
    cov_sub = cov[np.ix_(selected_idx, selected_idx)]
    
    min_eig = np.min(np.linalg.eigvalsh(cov_sub))
    if min_eig < 1e-10:
        cov_sub += np.eye(n) * (1e-10 - min_eig)
    
    bounds = [(0, min(max_weight, 1.0))] * n
    
    sub_tickers_list = [tickers[i] for i in selected_idx]
    base_constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
    if sector_limit < 1.0:
        for sec in SECTORS:
            sec_idx = [j for j, t in enumerate(sub_tickers_list) if SECTOR_MAP.get(t) == sec]
            if len(sec_idx) > 0:
                base_constraints.append({
                    'type': 'ineq',
                    'fun': lambda w, si=sec_idx, sl=sector_limit: sl - np.sum(w[si])
                })
    
    # Min and max feasible returns
    w0 = np.ones(n) / n
    
    # Min variance portfolio
    res_min = minimize(lambda w: w @ cov_sub @ w, w0, method='SLSQP',
                       bounds=bounds, constraints=base_constraints,
                       options={'ftol': 1e-12, 'maxiter': 2000})
    w_min = res_min.x if res_min.success else w0
    
    ret_min = w_min @ mu_sub
    ret_max = np.max(mu_sub)  # upper bound
    
    target_rets = np.linspace(ret_min, ret_max * 0.95, n_points)
    frontier_ret, frontier_vol = [], []
    
    for target in target_rets:
        cons = base_constraints + [{'type': 'eq', 'fun': lambda w, t=target: w @ mu_sub - t}]
        res = minimize(lambda w: w @ cov_sub @ w, w0, method='SLSQP',
                       bounds=bounds, constraints=cons,
                       options={'ftol': 1e-12, 'maxiter': 2000})
        if res.success:
            frontier_ret.append(res.x @ mu_sub)
            frontier_vol.append(np.sqrt(res.x @ cov_sub @ res.x * 252))
    
    return np.array(frontier_ret), np.array(frontier_vol)

print("✓ Optimizer engine ready")

✓ Optimizer engine ready


In [4]:
# ── §4  Visualization ──────────────────────────────────────

SECTOR_COLORS = {
    'Technology':'#2563eb', 'Financials':'#059669', 'Healthcare':'#dc2626',
    'Cons. Discr.':'#d97706', 'Cons. Staples':'#7c3aed', 'Industrials':'#6b7280',
    'Energy':'#a16207', 'Communication':'#db2777', 'Utilities':'#0891b2',
    'Real Estate':'#4f46e5', 'Materials':'#65a30d',
}

def plot_results(result, frontier_ret=None, frontier_vol=None, output_area=None):
    """Generate allocation chart, weights table, and efficient frontier."""
    w = result['weights']
    ticks = result['tickers']
    
    # Filter to non-zero weights
    mask = w > 0.001
    w_nz = w[mask]
    t_nz = [ticks[i] for i in range(len(ticks)) if mask[i]]
    sectors_nz = [SECTOR_MAP.get(t, 'Other') for t in t_nz]
    colors_nz = [SECTOR_COLORS.get(s, '#999') for s in sectors_nz]
    
    # Sort by weight descending
    order = np.argsort(w_nz)[::-1]
    w_nz = w_nz[order]
    t_nz = [t_nz[i] for i in order]
    sectors_nz = [sectors_nz[i] for i in order]
    colors_nz = [colors_nz[i] for i in order]
    
    fig = plt.figure(figsize=(14, 10))
    gs = fig.add_gridspec(2, 2, height_ratios=[1, 1], hspace=0.35, wspace=0.3)
    
    # ── Top left: Summary stats ──
    ax_text = fig.add_subplot(gs[0, 0])
    ax_text.axis('off')
    summary = (
        f"Expected Return:   {result['expected_return']*100:>7.2f}%\n"
        f"Expected Vol:      {result['expected_vol']*100:>7.2f}%\n"
        f"Sharpe Ratio:      {result['sharpe']:>7.2f}\n"
        f"Positions:         {int(np.sum(mask)):>7d}\n"
        f"Max Weight:        {np.max(w_nz)*100:>7.1f}%\n"
        f"Optimization:      {'✓ converged' if result['success'] else '✗ fallback (1/N)'}"
    )
    ax_text.text(0.05, 0.95, "Portfolio Summary", fontsize=14, fontweight='bold',
                 transform=ax_text.transAxes, va='top', color='#0f1f3d')
    ax_text.text(0.05, 0.75, summary, fontsize=11, fontfamily='monospace',
                 transform=ax_text.transAxes, va='top', color='#1e293b',
                 linespacing=1.6)
    
    # ── Top right: Bar chart of weights ──
    ax_bar = fig.add_subplot(gs[0, 1])
    top_n = min(20, len(t_nz))
    y_pos = np.arange(top_n)
    ax_bar.barh(y_pos, w_nz[:top_n] * 100, color=colors_nz[:top_n], edgecolor='white', height=0.7)
    ax_bar.set_yticks(y_pos)
    ax_bar.set_yticklabels(t_nz[:top_n], fontsize=9)
    ax_bar.invert_yaxis()
    ax_bar.set_xlabel('Weight (%)')
    ax_bar.set_title(f'Top {top_n} Holdings', fontweight='bold')
    ax_bar.grid(axis='x', alpha=0.2)
    
    # ── Bottom left: Sector allocation pie ──
    ax_pie = fig.add_subplot(gs[1, 0])
    sec_weights = {}
    for i, t in enumerate(ticks):
        if w[i] > 0.001:
            sec = SECTOR_MAP.get(t, 'Other')
            sec_weights[sec] = sec_weights.get(sec, 0) + w[i]
    
    sec_sorted = sorted(sec_weights.items(), key=lambda x: -x[1])
    labels = [f"{s[0]} ({s[1]*100:.1f}%)" for s in sec_sorted]
    sizes  = [s[1] for s in sec_sorted]
    colors_pie = [SECTOR_COLORS.get(s[0], '#999') for s in sec_sorted]
    
    ax_pie.pie(sizes, labels=labels, colors=colors_pie, startangle=90,
               textprops={'fontsize': 8}, pctdistance=0.8,
               wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
    ax_pie.set_title('Sector Allocation', fontweight='bold')
    
    # ── Bottom right: Efficient frontier ──
    ax_ef = fig.add_subplot(gs[1, 1])
    if frontier_ret is not None and len(frontier_ret) > 2:
        ax_ef.plot(frontier_vol * 100, frontier_ret * 100, '-', color='#2563eb',
                   lw=2, label='Efficient Frontier')
    ax_ef.scatter(result['expected_vol'] * 100, result['expected_return'] * 100,
                  color='#dc2626', s=120, zorder=5, edgecolors='white', lw=2,
                  label='Selected Portfolio')
    ax_ef.set_xlabel('Annualized Volatility (%)')
    ax_ef.set_ylabel('Annualized Return (%)')
    ax_ef.set_title('Efficient Frontier', fontweight='bold')
    ax_ef.legend(fontsize=9)
    ax_ef.grid(True, alpha=0.2)
    
    plt.suptitle('M6 — Portfolio Optimizer Results', fontsize=15, fontweight='bold',
                 color='#0f1f3d', y=1.01)
    plt.tight_layout()
    plt.show()
    
    # ── Weights table ──
    df_w = pd.DataFrame({
        'Ticker': t_nz, 'Sector': sectors_nz,
        'Weight (%)': [f"{x*100:.2f}" for x in w_nz]
    })
    display(HTML("<h4 style='color:#0f1f3d;'>Full Weights Table</h4>"))
    display(df_w.style.set_properties(**{'text-align':'left'}).hide(axis='index'))

print("✓ Visualization functions ready")

✓ Visualization functions ready


---
## Control Panel

Adjust the settings below and click **Optimize** to compute your portfolio.

In [ ]:
# ── §5  Interactive Control Panel ───────────────────────────

# Build display labels: "AAPL — Apple Inc."
_ticker_labels = []
for t in tickers:
    name = _meta.loc[t, 'name'] if t in _meta.index else t
    if len(name) > 30:
        name = name[:28] + '…'
    _ticker_labels.append(f"{t:<5} — {name}")

_label_to_ticker = {lbl: t for lbl, t in zip(_ticker_labels, tickers)}

# ── Estimator dropdown ──
w_estimator = widgets.Dropdown(
    options=estimator_names,
    value=estimator_names[0],
    description='Estimator:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px'),
)

# ── Risk tolerance ──
w_risk = widgets.BoundedFloatText(
    value=0.0, min=0.0, max=1.0, step=0.05,
    description='Risk Tolerance:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='280px'),
)
w_risk_help = widgets.HTML(
    "<span style='color:#6b7280; font-size:0.85em;'>"
    "0 = min variance &nbsp;·&nbsp; 1 = max return</span>"
)

# ── Lookback window ──
w_lookback = widgets.BoundedIntText(
    value=252, min=60, max=504, step=21,
    description='Lookback (days):',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='280px'),
)

# ── Max position size ──
w_maxpos = widgets.BoundedFloatText(
    value=1.0, min=0.02, max=1.0, step=0.01,
    description='Max Weight:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='280px'),
)

# ── Sector limit ──
w_seclimit = widgets.BoundedFloatText(
    value=1.0, min=0.10, max=1.0, step=0.05,
    description='Sector Limit:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='280px'),
)

# ── Show efficient frontier checkbox ──
w_show_ef = widgets.Checkbox(
    value=True, description='Show Efficient Frontier',
    style={'description_width': 'auto'},
)

# ── Stock selection: SelectMultiple with monospace font ──
w_stocks = widgets.SelectMultiple(
    options=_ticker_labels,
    value=_ticker_labels,
    description='',
    layout=widgets.Layout(width='420px', height='320px'),
)

display(HTML("""
<style>
  .widget-select-multiple select {
    font-family: 'Courier New', Courier, monospace;
    font-size: 13px;
  }
</style>
"""))

w_select_all = widgets.Button(
    description='Select All',
    button_style='',
    layout=widgets.Layout(width='110px', height='30px'),
)
w_clear_all = widgets.Button(
    description='Clear All',
    button_style='warning',
    layout=widgets.Layout(width='110px', height='30px'),
)

def _select_all(_):
    w_stocks.value = _ticker_labels

def _clear_all(_):
    w_stocks.value = []

w_select_all.on_click(_select_all)
w_clear_all.on_click(_clear_all)

stock_help = widgets.HTML(
    "<span style='color:#6b7280; font-size:0.82em;'>"
    "Ctrl/⌘-click to toggle individual stocks &nbsp;·&nbsp; Shift-click for a range.</span>"
)

# ── Optimize button ──
w_button = widgets.Button(
    description='  Optimize',
    button_style='primary',
    icon='calculator',
    layout=widgets.Layout(width='180px', height='40px'),
)

# ── Output area ──
output = widgets.Output()

def on_optimize(btn):
    with output:
        clear_output(wait=True)

        selected = [_label_to_ticker[lbl] for lbl in w_stocks.value
                    if lbl in _label_to_ticker]

        if len(selected) < 2:
            print("⚠ Select at least 2 stocks.")
            return

        lookback   = w_lookback.value
        ret_window = returns.iloc[-lookback:]

        print(f"Running: {w_estimator.value} | {len(selected)} stocks | "
              f"{lookback}-day window | risk={w_risk.value:.2f}")
        print("Computing covariance matrix...")

        est_func = ESTIMATORS[w_estimator.value]
        try:
            cov_full = est_func(ret_window)
        except Exception as e:
            print(f"✗ Estimator error: {e}")
            print("  Falling back to sample covariance.")
            cov_full = ret_window.cov().values

        mu = estimate_mu(ret_window)

        result = optimize_portfolio(
            mu, cov_full,
            risk_tolerance=w_risk.value,
            max_weight=w_maxpos.value,
            sector_limit=w_seclimit.value,
            selected_tickers=selected,
            all_tickers=tickers,
        )

        if result is None:
            print("✗ Optimization failed — no valid tickers selected.")
            return

        frontier_ret, frontier_vol = None, None
        if w_show_ef.value:
            print("Computing efficient frontier...")
            sel_idx = [tickers.index(t) for t in selected]
            try:
                frontier_ret, frontier_vol = compute_efficient_frontier(
                    mu, cov_full, sel_idx,
                    max_weight=w_maxpos.value,
                    sector_limit=w_seclimit.value,
                )
            except Exception as e:
                print(f"  ⚠ Frontier computation failed: {e}")

        print("Done.\n")
        plot_results(result, frontier_ret, frontier_vol)

w_button.on_click(on_optimize)

# ── Layout ──
controls_left = widgets.VBox([
    widgets.HTML("<h3 style='color:#0f1f3d; margin:0 0 10px 0;'>Optimization Settings</h3>"),
    w_estimator,
    w_risk, w_risk_help,
    w_lookback,
    w_maxpos,
    w_seclimit,
    w_show_ef,
    w_button,
], layout=widgets.Layout(width='420px', padding='10px'))

controls_right = widgets.VBox([
    widgets.HTML("<h3 style='color:#0f1f3d; margin:0 0 6px 0;'>Stock Selection</h3>"),
    widgets.HBox([w_select_all, w_clear_all]),
    stock_help,
    w_stocks,
], layout=widgets.Layout(padding='10px'))

panel = widgets.HBox([controls_left, controls_right],
    layout=widgets.Layout(border='2px solid #0f1f3d', border_radius='6px',
                          padding='15px', margin='10px 0'))

display(panel)
display(widgets.HTML("<hr>"))
display(output)


---
<p style='text-align:center; color:#8fadc8; font-style:italic; font-size:0.85em;'>
Module 6 complete &mdash; Portfolio Optimizer Tool.
</p>